# Calculate the extremes
for: SMYLE and other products

In [ ]:
# general use packages
%matplotlib inline
import matplotlib.pyplot as plt
import xarray as xr
import pandas as pd
import numpy as np

# packages for altering time to match up!
import sys
import cftime

# climpred packages
import climpred
from climpred import HindcastEnsemble
from climpred.tutorial import load_dataset
from climpred.stats import rm_poly

# SMYLE Utility functions
from SMYLEutils import io_utils as io
from SMYLEutils import calendar_utils as cal
from SMYLEutils import stat_utils as stat

In [ ]:
def detrend_linear(dat, dim, deg):
    """ linear detrend dat along the axis dim """
    params = dat.polyfit(dim=dim, deg=1)
    fit = xr.polyval(dat[dim], params.polyfit_coefficients)
    dat = dat-fit
    return dat

In [ ]:
var = 'dRHO' # photoC_tot_zint_100m , totChl_zint_100m , totC_zint_100m, 100m_OHC
depth = 'surface'
str_init = '11'

In [ ]:
%time smyle = xr.open_dataset('/glade/derecho/scratch/smogen/Chl_extremes/monthly/SMYLE/' + var +  '.monthly.' + depth + '.' + str_init + '.regrid.nc')[var]
%time smyle_time = xr.open_dataset('/glade/derecho/scratch/smogen/Chl_extremes/monthly/SMYLE/photoC_tot_zint_100m.monthly.' + str_init + '.time.nc')

In [ ]:
%%time
smyle_anom,smyle_clim = stat.remove_drift(smyle,smyle_time,1950,2019)

In [ ]:
smyle_anom = smyle_anom.time

### threshold within SMYLE

In [ ]:
thold = 0.9

In [ ]:
%%time
ds_thold = []

for i in range(len(smyle_anom.L)):
    tst = smyle_anom.isel(L=i).quantile(thold,dim=('M','Y'),skipna=True)
    tst = tst.expand_dims('L')
    ds_thold.append(tst)

In [ ]:
smyle_threshold = xr.concat(ds_thold,dim='L')

smyle_threshold['L'] = smyle_threshold.L + 1

In [ ]:
smyle_threshold = smyle_threshold.to_dataset(name='time').rename({'time':'threshold'})

In [ ]:
# save out the extremes
smyle_threshold.to_netcdf('/glade/derecho/scratch/smogen/Chl_extremes/monthly/SMYLE/thresholds/' + var + '.' + str_init + '.' + str(thold) + '.final.nc')

## extremes in SMYLE

In [ ]:
if thold == 0.9:
    smyle_extreme = smyle_anom.where(smyle_anom > smyle_threshold.threshold)
elif thold == 0.1:
    smyle_extreme = smyle_anom.where(smyle_anom < smyle_threshold.threshold)

In [ ]:
# save out the extremes!
(~np.isnan(smyle_extreme)).to_netcdf('/glade/derecho/scratch/smogen/Chl_extremes/monthly/SMYLE/extremes/' + var +  '.monthly.' + str_init + '.regrid.extremes.0.9.final.nc')